In [1]:
import tensorflow as tf
import keras
import matplotlib.pyplot as plt
import numpy as np

# Get to coding

## Cell 1 - Initial setup
We start by defining our dimensions and batch sizes for inceptionV3, in order for it to work properly according to its documentation.

In [2]:
# Defining image dimensions and batch size
IMG_SIZE = (299, 299) # InceptionV3 expects inputs of at least 75x75, but 299x299 is standard for its pre-trained weights.
BATCH_SIZE = 8
VAL_SPLIT = 0.33
EPOCHS = 20

DATA_DIR_TRAIN = './grocery_dataset/train'
print("Training data is located at ", DATA_DIR_TRAIN)

DATA_DIR_VAL = './grocery_dataset/val'
print("Validation data is located at ", DATA_DIR_VAL)

DATA_DIR_TEST = './grocery_dataset/test'
print("Validation data is located at ", DATA_DIR_TEST)

Training data is located at  ./grocery_dataset/train
Validation data is located at  ./grocery_dataset/val
Validation data is located at  ./grocery_dataset/test


## Splitting the data

### Cell 2 - Datasplit for training

In [3]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR_TRAIN,
    labels='inferred',
    label_mode='int',
    validation_split=VAL_SPLIT,
    subset='training',
    seed=1337,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

Found 15001 files belonging to 15 classes.
Using 10051 files for training.


### Cell 3 - Datasplit for validation

In [4]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR_VAL,
    labels='inferred',
    label_mode='int',
    validation_split=VAL_SPLIT,
    subset='validation',
    seed=1337,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

Found 3000 files belonging to 15 classes.
Using 990 files for validation.


### Cell 4 - Class naming

In [5]:
class_names = train_ds.class_names
num_classes = len(class_names)
print(f"Total number of classes: {num_classes}")
print(f"Product categories: {class_names}")

Total number of classes: 15
Product categories: ['Bean', 'Bitter_Gourd', 'Bottle_Gourd', 'Brinjal', 'Broccoli', 'Cabbage', 'Capsicum', 'Carrot', 'Cauliflower', 'Cucumber', 'Papaya', 'Potato', 'Pumpkin', 'Radish', 'Tomato']


## Cell 5 - Optimizing for performance
Adding caching is a good idea, since we dont want the GPU or CPU to wait for data loading.

In [6]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

### Cell 6 - Generating true labels

In [7]:
y_true = []
for images, labels in val_ds:
    y_true.extend(labels.numpy())

y_true = np.array(y_true)

2026-05-28 12:57:14.018925: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## Setting up the model
###  Cell 7 - Loading the pretrained base of InceptionV3
Using InceptionV3'd ImageNet pre-trained dataset, but excluding the classification layer.

In [8]:
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras import layers, models

# Instantiate the base model
base_model = InceptionV3(
    input_shape=IMG_SIZE + (3,), # Expected input shape
    include_top=False,           # Excluding the 1000-class classifier layer
    weights='imagenet'           # Use weights pre-trained on ImageNet
)

### Cell 8 - Freezing the base model
We freeze the weights of the base of InceptionV3, preventing the random training on our dataset from destroying the base knowledge and patterns learned from ImageNet.

In [9]:
base_model.trainable = False # FREEZE the layers

### Cell 9 - Adding our own little network
We add our own little network (a new head) on top of the frozen base. This head is responsible for taking the features extracted by InceptionV3  and classifying them into our product categories (num_classes).

In [10]:
# Reducing the 3D feature map to a 1D vector
global_average_layer = layers.GlobalAveragePooling2D() 
# Our final classification layer
prediction_layer = layers.Dense(num_classes, activation='softmax') 

data_augmentation = models.Sequential([
    # Flips images randomly
  layers.RandomFlip("horizontal_and_vertical"),
    # Rotates up to 20% (72 degrees)
  layers.RandomRotation(0.2),
    # Zooms in/out by 20%
  layers.RandomZoom(0.2),                         
])

# Combine the base and the head using the Functional API
model = models.Sequential([
    # Add the augmentation
    data_augmentation,

    # Add a preprocessing layer for InceptionV3's required input scaling
    # (if not handled by the data loader)
    layers.Lambda(tf.keras.applications.inception_v3.preprocess_input, input_shape=(299, 299, 3)),
    base_model,
    global_average_layer,
    layers.Dropout(0.2), # Good practice to prevent overfitting
    prediction_layer
])

model.build(input_shape=(None, 299, 299, 3))

# Print the summary to see your new, smaller model head
model.summary()

model.save("incep_model.keras")
model.save('saved_model_dir.keras')

/home/bram/.local/lib/python3.11/site-packages/keras/src/layers/core/lambda_layer.py:65: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 299, 299, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 299, 299, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ inception_v3 (Functional)       │ (None, 8, 8, 2048)     │    21,802,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 15)             │        30,735 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,833,519 (83.29 MB)

 Trainable params: 30,735 (120.06 KB)

 Non-trainable params: 21,802,784 (83.17 MB)

### Cell 10 - Optimizer
Compiling with adam optimizer to up the performance a little. Adam combines the best features of two optimizers i.e Momentum and RMSprop.

In [11]:
from tensorflow.keras.optimizers import Adam

optimizer = Adam(learning_rate=0.0001)
model.compile(optimizer=optimizer, loss=keras.losses.SparseCategoricalCrossentropy(), metrics=['accuracy'])

## Time to train!
### Cell 11 - Training
Training without fine tuning first...

In [ ]:
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
)

Epoch 1/20
 615/1257 ━━━━━━━━━━━━━━━━━━━━ 1:29 139ms/step - accuracy: 0.4060 - loss: 2.0715

In [ ]:
# Extract data from the history object
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

# Plot the training and validation accuracy
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

# Plot the training and validation loss
plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

print("Training accuracy: ", acc)
print("Validation accuracy: ", val_acc)

In [ ]:
## Stress testing the model
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.inception_v3 import preprocess_input

def TestAccuracy(path):
    img_path = path
    # Loading the image and resizing it to 299x299 (InceptionV3 standard)
    img = image.load_img(img_path, target_size=(299, 299))
    # Converting image to a numpy array and adding a "batch" dimension
    # (Keras expects a batch of images, even if it's just one)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    # Running the prediction
    predictions = model.predict(img_array)
    # Getting the class with the highest probability
    # score = tf.nn.softmax(predictions[0])
    score = predictions[0]
    predicted_class = class_names[np.argmax(score)]
    predicted_class_index = np.argmax(predictions[0])
    confidence = np.max(predictions[0]) * 100
    #for p in predictions:
    #    print(p / 100)
    
    print(f"I am {confidence:.2f}% sure this is a(n) {predicted_class} ({predicted_class_index}).")

TestAccuracy('./grocery_dataset/test/Tomato/1080.jpg')
TestAccuracy('./grocery_dataset/test/Cabbage/1080.jpg')
TestAccuracy('./grocery_dataset/test/Cucumber/1080.jpg')
TestAccuracy('./grocery_dataset/test/Carrot/1080.jpg')
TestAccuracy('./grocery_dataset/test/Radish/1080.jpg')
TestAccuracy('./grocery_dataset/test/Papaya/1200.jpg')

### Cell 12 - Converting the model

Converting the trained InceptionV3 model is done using the script from ./incep_conversion.py